# CUB — Data Analysis
Data only (no model). Works for **full CUB and CUB70** — set `PKLS_DIR` in the first cell.
Class balance · attribute prevalence · class×attribute matrix · **species-constancy** (does CUB vary within a species? → whether the recall gap is powered here) · candidate attribute-group features.

In [ ]:
import os, sys, pickle
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CURATED = Path(os.environ["CURATED_DATA"]); REPO = Path.cwd().parent   # run from curated/notebooks
# ---- CONFIG: switch dataset here ----
PKLS_DIR = CURATED / "CUB_processed" / "class_attr_data_10"                 # full CUB (200)
# PKLS_DIR = CURATED / "CUB_processed" / "class_attr_data_10_cub70_original"  # CUB70 (70)
ATTR_DIR = CURATED / "CUB_200_2011"
# -------------------------------------
def load(f):
    p = PKLS_DIR / f
    return pickle.load(open(p,"rb")) if p.exists() else None
tr = load("train.pkl"); te = load("test.pkl")
Atr=np.array([r["attribute_label"] for r in tr]); ytr=np.array([r["class_label"] for r in tr])
Ate=np.array([r["attribute_label"] for r in te]); yte=np.array([r["class_label"] for r in te])
nC=Atr.shape[1]
print(f"train {len(tr)} / test {len(te)} imgs · {len(set(ytr))} species · {nC} attributes  ({PKLS_DIR.name})")

# optional attribute names + groups (112 used attrs -> 28 groups)
names=groups=None
try:
    sys.path.insert(0, str(REPO/"external"/"minimal_cbm"))
    from src.datasets.cub200 import USED_ATTRIBUTES
    alln=[l.split(" ",1)[1].strip() for l in open(ATTR_DIR/"attributes.txt") if l.strip()]
    names=[alln[i-1] for i in USED_ATTRIBUTES]
    groups={}; [groups.setdefault(n.split("::")[0],[]).append(j) for j,n in enumerate(names)]
    print(f"{len(names)} attrs in {len(groups)} groups")
except Exception as e:
    print("names/groups unavailable (indices only):", e)

## 1. Class balance

In [ ]:
tc=pd.Series(ytr).value_counts().sort_index(); ec=pd.Series(yte).value_counts().sort_index()
print("train/species:",tc.min(),"-",tc.max(),"| test/species:",ec.min(),"-",ec.max())
fig,ax=plt.subplots(1,2,figsize=(10,2.6))
ax[0].bar(tc.index,tc.values); ax[0].set_title("train / species")
ax[1].bar(ec.index,ec.values,color="tab:orange"); ax[1].set_title("test / species"); plt.tight_layout()

## 2. Attribute prevalence

In [ ]:
prev=Atr.mean(0)
fig,ax=plt.subplots(figsize=(12,3)); ax.bar(range(nC),prev); ax.set_ylabel("P(=1)"); ax.set_xlabel("attribute")
print("prevalence: min %.3f  max %.3f  mean %.3f"%(prev.min(),prev.max(),prev.mean()))

## 3. Class × attribute matrix

In [ ]:
M=pd.DataFrame(Atr).assign(c=ytr).groupby("c").mean().values
print("cells exactly 0/1:",round(float(np.mean((M==0)|(M==1))),4),"(1.0 = class-level/constant labels)")
fig,ax=plt.subplots(figsize=(11,5)); im=ax.imshow(M,aspect="auto",cmap="magma",vmin=0,vmax=1)
ax.set_xlabel("attribute"); ax.set_ylabel("species"); fig.colorbar(im,ax=ax,fraction=0.02)

## 4. Species-constancy — is the recall gap powered on CUB?
Within-species std of each attribute on test. **>0 ⇒ attributes vary within a species ⇒ matched-pair recall gap has real signal** (unlike FunnyBirds, where it is ~0 = n-per-species noise). If ~0 here, the CUB pkls use class-level labels and the recall gap is underpowered too.

In [ ]:
within=np.array([Ate[yte==c].std(0) for c in np.unique(yte)])
frac0=float(np.mean(within==0)); nimg=int(np.median([(yte==c).sum() for c in np.unique(yte)]))
print(f"(species,attr) with within-species std==0: {frac0:.4f} | mean within-species std {within.mean():.4g} | median test imgs/species {nimg}")
print("VERDICT:", "within-species variation EXISTS -> recall gap is testable on CUB (image-level labels)" if frac0<0.99
      else "species-constant (class-level labels) -> recall gap underpowered here too; use deletion/swap")

## 5. Attribute-group features — candidates vs measured backwash
Neutral per-group properties (only if names loaded). Line up against per-group backwash later; no causal claim.

In [ ]:
if groups:
    rows=[{"group":g,"n_attrs":len(ix),"mean_prev":round(float(Atr[:,ix].mean()),3)} for g,ix in groups.items()]
    prof=pd.DataFrame(rows).sort_values("n_attrs",ascending=False); display(prof.head(30))
else:
    print("no group names -> skip (attribute indices only)")